# 🏥 Monte Carlo Prediction in a Simple Emergency Room Environment

This notebook demonstrates a simplified version of a custom emergency room simulation environment. It includes step-by-step **First-Visit** and **Every-Visit Monte Carlo** methods with debug outputs and basic visualization of state values.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
from collections import defaultdict
import matplotlib.pyplot as plt

class SimpleEmergencyRoomEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(2)  # 0=treat, 1=wait
        self.observation_space = spaces.Discrete(3)  # 0=few, 1=some, 2=none (patients)
        self.reset()

    def reset(self, seed=None, options=None):
        self.patients = 2
        self.time = 0
        return self.patients, {}

    def step(self, action):
        reward = 0
        if action == 0 and self.patients > 0:
            self.patients -= 1
            reward = 10
        elif action == 1:
            reward = -1
        else:
            reward = -5
        self.time += 1
        done = self.patients == 0 or self.time >= 5
        return self.patients, reward, done, False, {}

    def render(self):
        print(f"Time: {self.time}, Patients: {self.patients}")

## 🧪 Generate Episode with Debugging

In [ ]:
def generate_episode(env):
    episode = []
    state, _ = env.reset()
    done = False
    while not done:
        action = random.choice([0, 1])
        next_state, reward, done, _, _ = env.step(action)
        episode.append((state, action, reward))
        print(f"State: {state}, Action: {action}, Reward: {reward}, Next state: {next_state}")
        state = next_state
    return episode

## 🧠 First-Visit Monte Carlo with Debug Output

In [ ]:
def first_visit_mc(env, episodes=100, gamma=0.9):
    returns = defaultdict(list)
    V = defaultdict(float)
    for i in range(episodes):
        print(f"\nEpisode {i+1}")
        episode = generate_episode(env)
        G = 0
        visited = set()
        for t in reversed(range(len(episode))):
            state, _, reward = episode[t]
            G = gamma * G + reward
            if state not in visited:
                returns[state].append(G)
                V[state] = np.mean(returns[state])
                visited.add(state)
    return V

## 🔁 Every-Visit Monte Carlo with Debug Output

In [ ]:
def every_visit_mc(env, episodes=100, gamma=0.9):
    returns = defaultdict(list)
    V = defaultdict(float)
    for i in range(episodes):
        print(f"\nEpisode {i+1}")
        episode = generate_episode(env)
        G = 0
        for t in reversed(range(len(episode))):
            state, _, reward = episode[t]
            G = gamma * G + reward
            returns[state].append(G)
            V[state] = np.mean(returns[state])
    return V

## 📊 Compare Both Methods with Plot

In [ ]:
env = SimpleEmergencyRoomEnv()
V_first = first_visit_mc(env)
V_every = every_visit_mc(env)

states = sorted(set(V_first.keys()).union(V_every.keys()))
first_vals = [V_first.get(s, 0) for s in states]
every_vals = [V_every.get(s, 0) for s in states]

plt.figure(figsize=(8,4))
plt.plot(states, first_vals, label='First-Visit', marker='o')
plt.plot(states, every_vals, label='Every-Visit', marker='s')
plt.xlabel('State (Patients)')
plt.ylabel('Estimated Value')
plt.title('Monte Carlo Value Estimates')
plt.legend()
plt.grid(True)
plt.show()